In [1]:
!curl -A "Mozilla/5.0" -L -o /content/development.mp4 https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5889k  100 5889k    0     0  14.7M      0 --:--:-- --:--:-- --:--:-- 14.7M


In [3]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 9.9 MB/s eta 0:00:00


In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
import importlib.util
import subprocess
import sys

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
missing = [
    name for name in ("onnx", "onnxruntime")
    if importlib.util.find_spec(name) is None
]

if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])

In [6]:
VIDEO_PATH = "/content/development.mp4"
MODEL_PATH = "yolo26n.pt"
TRACKER_PATH = "bytetrack.yaml"

START_FRAME = 500
N_FRAMES = 100
WARMUP_RUNS = 5

PREDICT_ARGS = dict(
    imgsz=640,
    rect=False,
    half=False,
    classes=[0, 1, 2],
    conf=0.10,
    iou=0.70,
    verbose=False,
    save=False
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else "Unavailable"
)
print(f"Measured source frames: {START_FRAME}–{START_FRAME + N_FRAMES - 1}")

GPU: Tesla T4
Measured source frames: 500–599


**Export ONNX**

In [7]:
ONNX_PATH = YOLO(MODEL_PATH).export(
    format="onnx",
    imgsz=640,
    batch=1,
    dynamic=False,
    half=False,
    simplify=False,
    opset=17,
    device="cpu"
)

ONNX_PATH = str(ONNX_PATH)
print("Exported:", ONNX_PATH)

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.158 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.3 MB)

ONNX: starting export with onnx 1.23.0 opset 17...
ONNX: export success ✅ 1.1s, saved as 'yolo26n.onnx' (9.4 MB)

Export complete (1.9s)
Results saved to /content/yolo26n.onnx
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app
Exported: yolo26n.onnx
